# Milestone 1: Data Cleansing, Veracity Check & Storage Benchmark
**Kelompok Big Data - E-Commerce & Retail Analytics**

Notebook ini mencakup:
1. Ingest 3 entitas dataset terintegrasi (`transactions.csv`, `customers.csv`, `clickstream.jsonl`).
2. Deteksi anomali data kotor (Missing values, Inconsistent string formatting, Outliers/invalid values, Duplikasi).
3. Eksekusi data cleansing pipeline.
4. Storage footprint benchmark dan query latency benchmark (CSV vs JSONL vs Parquet).

In [ ]:
import os
import time
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

con = duckdb.connect(database=':memory:')
print('DuckDB Engine Initialized!')

## 1. Profiling Raw Data & Veracity Check (5% - 10% Anomali)

In [ ]:
raw_csv = '../generated_data/transactions_raw.csv'
df_sample = con.execute(f"SELECT * FROM read_csv('{raw_csv}') LIMIT 5").fetchdf()
df_sample

In [ ]:
# Deteksi Anomali Kotor
total_rows = con.execute(f"SELECT COUNT(*) FROM read_csv('{raw_csv}', all_varchar=true)").fetchone()[0]
null_cust = con.execute(f"SELECT COUNT(*) FROM read_csv('{raw_csv}', all_varchar=true) WHERE customer_id IS NULL OR customer_id = ''").fetchone()[0]
null_cat = con.execute(f"SELECT COUNT(*) FROM read_csv('{raw_csv}', all_varchar=true) WHERE category IS NULL OR category = ''").fetchone()[0]
neg_price = con.execute(f"SELECT COUNT(*) FROM read_csv('{raw_csv}') WHERE price < 0").fetchone()[0]
duplicates = con.execute(f"SELECT COUNT(*) FROM (SELECT transaction_id FROM read_csv('{raw_csv}') GROUP BY transaction_id HAVING COUNT(*) > 1)").fetchone()[0]

print(f"Total Raw Ingested: {total_rows:,}")
print(f"Missing Customer IDs: {null_cust:,}")
print(f"Missing Categories: {null_cat:,}")
print(f"Negative Outliers (Price): {neg_price:,}")
print(f"Duplicate Transaction IDs: {duplicates:,}")

## 2. Cleansing Pipeline Implementation

In [ ]:
clean_query = f"""
CREATE OR REPLACE TABLE clean_transactions AS
SELECT 
    transaction_id,
    CAST(customer_id AS BIGINT) AS customer_id,
    CAST(timestamp AS TIMESTAMP) AS timestamp,
    product_id,
    category,
    CAST(quantity AS INTEGER) AS quantity,
    CAST(price AS DOUBLE) AS price,
    CAST(discount AS DOUBLE) AS discount,
    payment_method,
    CASE 
        WHEN lower(city) LIKE '%jkt%' OR lower(city) LIKE '%jakarta%' THEN 'Jakarta'
        WHEN lower(city) LIKE '%sby%' OR lower(city) LIKE '%surabaya%' THEN 'Surabaya'
        WHEN lower(city) LIKE '%bdg%' OR lower(city) LIKE '%bandung%' THEN 'Bandung'
        WHEN lower(city) LIKE '%mdn%' OR lower(city) LIKE '%medan%' THEN 'Medan'
        WHEN lower(city) LIKE '%smg%' OR lower(city) LIKE '%semarang%' THEN 'Semarang'
        ELSE coalesce(city, 'Unknown')
    END AS city,
    ROUND(quantity * price * (1.0 - discount), 2) AS total_amount
FROM read_csv('{raw_csv}')
WHERE customer_id IS NOT NULL 
  AND category IS NOT NULL 
  AND timestamp IS NOT NULL
  AND price > 0
  AND quantity > 0
QUALIFY ROW_NUMBER() OVER(PARTITION BY transaction_id ORDER BY timestamp DESC) = 1
"""
con.execute(clean_query)
clean_count = con.execute("SELECT COUNT(*) FROM clean_transactions").fetchone()[0]
print(f"Cleansing Selesai! Data Bersih: {clean_count:,} baris ({clean_count/total_rows*100:.2f}% Veracity Health Score)")

## 3. Storage Footprint & Query Latency Benchmark (CSV vs JSONL vs Parquet)

In [ ]:
clean_csv = '../generated_data/transactions_clean.csv'
clean_parquet = '../generated_data/transactions_clean.parquet'
clean_jsonl = '../generated_data/transactions_clean.jsonl'

con.execute(f"COPY clean_transactions TO '{clean_parquet}' (FORMAT PARQUET, COMPRESSION 'SNAPPY')")
con.execute(f"COPY clean_transactions TO '{clean_csv}' (HEADER, DELIMITER ',')")
con.execute(f"COPY clean_transactions TO '{clean_jsonl}' (FORMAT JSON)")

csv_mb = os.path.getsize(clean_csv) / (1024 * 1024)
jsonl_mb = os.path.getsize(clean_jsonl) / (1024 * 1024)
parquet_mb = os.path.getsize(clean_parquet) / (1024 * 1024)

print(f"CSV Size: {csv_mb:.2f} MB")
print(f"JSONL Size: {jsonl_mb:.2f} MB")
print(f"Parquet Size: {parquet_mb:.2f} MB (Hemat {(1 - parquet_mb/csv_mb)*100:.1f}%)")

In [ ]:
# Benchmark Query Aggregation
def benchmark_sql(sql_func, name):
    t0 = time.perf_counter()
    con.execute(sql_func).fetchall()
    t1 = time.perf_counter()
    lat_ms = (t1 - t0) * 1000
    print(f"{name}: {lat_ms:.2f} ms")
    return lat_ms

q_csv = f"SELECT category, city, COUNT(*), SUM(total_amount) FROM read_csv('{clean_csv}') GROUP BY category, city"
q_json = f"SELECT category, city, COUNT(*), SUM(total_amount) FROM read_json('{clean_jsonl}') GROUP BY category, city"
q_parquet = f"SELECT category, city, COUNT(*), SUM(total_amount) FROM read_parquet('{clean_parquet}') GROUP BY category, city"

t_csv = benchmark_sql(q_csv, 'CSV Query')
t_json = benchmark_sql(q_json, 'JSONL Query')
t_parquet = benchmark_sql(q_parquet, 'Parquet Query')

print(f"Speedup Parquet vs CSV: {t_csv / max(t_parquet, 0.01):.1f}x lebih cepat!")